# Cost Controls in a LiteLLM proxy: a per-project budget that blocks runaway spend

An agent stuck in a retry loop, or a prompt that drags a huge context into every turn, can spend a
month of model budget in an afternoon. **Cost Controls** give each Highflame project a daily and a
monthly budget in US dollars. Highflame prices every model call it is told about, adds it to the
project's spend, and blocks new prompts once the budget is used up.

This notebook is for a **LiteLLM proxy** that calls every provider itself and asks Highflame about
each request through the guardrail in [`../../litellm/`](../../litellm/) (Mode B). One line in the
guardrail's config, `report_usage: true`, makes it report each call's token counts. The guardrail's
usual `pre_call` check then refuses the next prompt once the budget is used up. The apps and agents
behind the proxy do not change.

1. Read the project's budget, without spending anything.
2. Start a LiteLLM proxy with the guardrail, and make one call through it.
3. Keep going until the budget is used up, and see the next prompt refused before it reaches OpenAI.
4. See where the money went, by agent and by model.

Routes that send their calls through the Highflame AI Gateway (Mode A) need nothing: the gateway
reports and refuses on its own, as in [`01-ai-gateway`](../01-ai-gateway/). The guardrail skips
those calls, so they are never counted twice.

## How it works

1. A client sends a chat request to the LiteLLM proxy.
2. The guardrail's `pre_call` check sends the prompt to Highflame. This is where a used-up budget is
   refused.
3. LiteLLM calls the provider directly and answers the client.
4. Then LiteLLM hands the call's token counts to the guardrail, which reports them to Highflame.
   Highflame prices the call from one price table and adds it to the project's spend for the day and
   for the month (UTC).
5. On the **next** prompt, the `pre_call` check runs the Cost Controls policy, which compares the
   spend with the budget and refuses once it is used up.

The report goes out after the client has its answer, so it adds no latency. The price of a call is
known only after the model answers, so the call that crosses the budget completes and the one after
it is refused. If that overshoot matters, use the **Monthly Budget Headroom** template, which blocks
at 95% of the budget.

## Before you start: set it up once in Studio

Use a project that nothing else depends on, and only this notebook. When the budget is used up,
**every** prompt in the project is blocked until the next UTC day, or until you raise the budget.

1. Sign in at [studio.highflame.ai](https://studio.highflame.ai) and create a project, for example
   `cost-controls-litellm`.
2. **Custom Agents → Policies → Cost Controls → Set budget.** Set **Daily budget (USD)** to `0.05`, leave
   the monthly budget blank, and select **Deploy**. The budget belongs to the project, so the Custom
   Agents and AI Gateway pages show the same one.
3. In the same tab, deploy the **Daily Budget** template. The deploy dialog starts on **Monitor**,
   which only records; set **Enforcement Mode** to **Enforce**.
4. **Registry → Register Identity.** Keep **Agent Identity** on, name the agent (for example `LiteLLM Proxy`), and select **Register Identity**. Copy the API key it shows; Studio shows it only once. The proxy signs every guard check and usage report with it, so the spend is the agent's. Any other API key in the project also works, but then the chargeback cannot name the proxy.

With `gpt-4o` each call below costs about one cent, so a $0.05 budget runs out in a handful of
calls. That is the point: the notebook should hit the limit in front of you.

You also need Docker: the notebook builds and runs a LiteLLM proxy image. Check it with
`docker version`.

## Setup

Copy `.env.example` to `.env` beside this notebook.

| Variable | What it is |
| --- | --- |
| `HIGHFLAME_API_KEY` | **Required.** An API key in the project from step 1. Use the Registry agent's key from step 4, so the chargeback names the proxy. |
| `PROVIDER_API_KEY` | **Required.** Your OpenAI API key. LiteLLM calls OpenAI with it directly. `OPENAI_API_KEY` works as a fallback. |
| `PROXY_PORT` | Optional. The local port for the test proxy. Defaults to `4001`. |
| `MODEL` | An OpenAI model. Defaults to `gpt-4o`, which costs enough per call to reach a small budget quickly. |
| `DAILY_BUDGET_USD` | Optional. The notebook reads the budget amount from Highflame; this is used only if Highflame does not report it. Defaults to `0.05`. |

In [1]:
%pip install -q -r requirements.txt

from importlib.metadata import version

print("highflame", version("highflame"))

Note: you may need to restart the kernel to use updated packages.
highflame 0.10.0


In [2]:
import json
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.parse
import urllib.request
import uuid
from datetime import datetime, timedelta, timezone
from pathlib import Path

from dotenv import load_dotenv
from highflame import Highflame

load_dotenv()

HIGHFLAME_API_KEY = os.environ["HIGHFLAME_API_KEY"]
PROVIDER_API_KEY = os.environ.get("PROVIDER_API_KEY") or os.environ.get("OPENAI_API_KEY", "")
if not PROVIDER_API_KEY:
    raise RuntimeError("Set PROVIDER_API_KEY (or OPENAI_API_KEY) in .env to your OpenAI API key.")
MODEL = os.environ.get("MODEL", "gpt-4o")
DAILY_BUDGET_USD = float(os.environ.get("DAILY_BUDGET_USD", "0.05"))

# The platform API and the identity service. Leave both unset for the hosted product.
API_URL = os.environ.get("HIGHFLAME_API_URL", "https://api.highflame.ai").rstrip("/")
AUTH_URL = os.environ.get("HIGHFLAME_AUTH_URL", "https://auth.highflame.ai").rstrip("/")
SDK_URLS = {"base_url": API_URL, "identity_base_url": AUTH_URL, "token_url": f"{AUTH_URL}/oauth2/token"}

# The local test proxy. Set PROXY_PORT in .env if 4001 is busy.
PROXY_PORT = int(os.environ.get("PROXY_PORT", "4001"))
PROXY_URL = f"http://localhost:{PROXY_PORT}"
MASTER_KEY = "sk-local-cost-demo"  # stands in for your LiteLLM virtual key
IMAGE = "litellm-highflame-cost:demo"
CONTAINER = "hf-litellm-cost"
WORK = Path("_highflame_litellm_cost").resolve()  # build files, and the proxy's env file with your keys
GUARDRAIL = Path("../../litellm/mode_b_shield_guardrail.py")

RUN_ID = uuid.uuid4().hex[:6]


def sh(cmd: str) -> tuple[int, str]:
    # Run a shell command and return (return code, stdout + stderr).
    done = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


highflame = Highflame(api_key=HIGHFLAME_API_KEY, **SDK_URLS)
identity = highflame.guard.evaluate_prompt("who am I", mode="monitor").agent_identity
if identity is not None and identity.identity_type == "agent":
    print("key    : agent", identity.name, f"({identity.external_id}, {identity.status})")
else:
    print("key    : a project key, not a registered agent. The chargeback lists the proxy's spend")
    print("         as '(not a registered agent)'; a Registry agent's key names it (step 4 above).")
print("model  :", MODEL)
rc, docker_version = sh("docker version --format '{{.Server.Version}}'")
if rc != 0:
    raise RuntimeError("Docker is not running. Start it, then run this cell again.")
print("docker :", docker_version)

key    : agent LiteLLM Proxy (litellm-proxy, active)
model  : gpt-4o
docker : 28.3.0


### Today's spend so far

Highflame keeps one record per priced call and serves them as a chargeback report. Read today's
totals now, before this run spends anything, so section 4 can show what this run added.

In [3]:
form = urllib.parse.urlencode({"grant_type": "api_key", "api_key": HIGHFLAME_API_KEY}).encode()
with urllib.request.urlopen(
    urllib.request.Request(f"{AUTH_URL}/oauth2/token", data=form, method="POST",
                           headers={"Content-Type": "application/x-www-form-urlencoded"}),
    timeout=30,
) as response:
    read_token = json.loads(response.read())["access_token"]


def chargeback(group_by: str) -> dict:
    now = datetime.now(timezone.utc)
    query = urllib.parse.urlencode({
        "start": now.strftime("%Y-%m-%dT00:00:00Z"),
        "end": (now + timedelta(minutes=1)).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "group_by": group_by,
    })
    request = urllib.request.Request(
        f"{API_URL}/v1/obs/costs/chargeback?{query}", headers={"Authorization": f"Bearer {read_token}"}
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        return json.loads(response.read())


BASELINE = chargeback("agent")
CALLS_MADE = 0  # priced calls this run makes; section 4 waits until the report has all of them
print(f"Today so far: ${BASELINE.get('total_cost_usd', 0):.4f} over {BASELINE.get('total_calls', 0)} calls")

Today so far: $0.0000 over 0 calls


## 1. Read the budget, without spending anything

`budget_state()` asks Highflame how much of the project's budget is used, the same way a policy
sees it. It runs a guard check in `monitor` mode with `debug=True`, which returns the values the
Cost Controls policies read:

| Value | Meaning |
| --- | --- |
| `budget_cost_pct_daily` | Share of today's budget used, 0 to 100 |
| `budget_exceeded_daily` | `True` once today's budget is used up |
| `agent_spend_cents_daily` | What the calling agent spent today, in US cents |

They appear only when they apply: the budget values only when the project has a daily budget, and
the agent value only when the caller is a registered agent.

The budget amount itself comes from Highflame too, through a `dryrun` guard check that returns the
configuration of every detector the active policies need. Edit the budget in Studio and the next
read shows the new amount.

The proxy needs nothing from this notebook. The notebook uses the SDK only to read the budget and
the chargeback report.

In [4]:
from highflame import GuardRequest

BUDGET_KEYS = (
    "budget_cost_pct_daily",
    "budget_exceeded_daily",
    "budget_cost_pct_monthly",
    "budget_exceeded_monthly",
    "agent_spend_cents_daily",
    "agent_spend_cents_monthly",
)


def budget_limits(client: Highflame = highflame) -> dict:
    # A dry run executes nothing: it returns the detectors the active policies need, with this
    # project's configuration for each. The budget checker's configuration is the budget.
    plan = client.guard.evaluate(GuardRequest(
        content="budget check", content_type="prompt", action="process_prompt", optimize=True, dryrun=True,
    )).optimization
    for detector in (plan.model_dump() if plan else {}).get("required_detectors") or []:
        if detector.get("name") == "budget_checker":
            return detector.get("config") or {}
    return {}


def budget_state(client: Highflame = highflame) -> dict:
    # mode="monitor": a read, never a block. The check costs nothing; it carries no tokens.
    decision = client.guard.evaluate_prompt("budget check", mode="monitor", debug=True)
    seen = {**(decision.context or {}), **(decision.projected_context or {})}
    state = {key: seen[key] for key in BUDGET_KEYS if key in seen}
    state["daily_usd"] = budget_limits(client).get("daily_usd", DAILY_BUDGET_USD)
    return state


def meter(state: dict) -> str:
    pct = int(state.get("budget_cost_pct_daily", 0))
    bar = "#" * (pct // 5) + "." * (20 - pct // 5)
    budget = state.get("daily_usd") or DAILY_BUDGET_USD
    used = budget * pct / 100
    flag = "  USED UP" if state.get("budget_exceeded_daily") else ""
    return f"[{bar}] {pct:3d}% of ${budget:.2f} (~${used:.3f}){flag}"


state = budget_state()
if "budget_cost_pct_daily" not in state:
    raise RuntimeError(
        "This project has no daily budget. Do step 2 of 'Before you start' in Studio, then run this cell again."
    )
print("project today:", meter(state))
if state.get("budget_exceeded_daily"):
    print(
        "\nToday's budget is already used up, probably by an earlier run. Raise the budget in Studio"
        " (Cost Controls -> Edit budget), or run this again after 00:00 UTC."
    )

project today: [....................]   0% of $0.05 (~$0.000)


## 2. A LiteLLM proxy that reports its usage

### Build the image

The image is the stock LiteLLM proxy with the Highflame SDK added and the guardrail module copied in,
the same as in the guardrail recipe. `usage.report()` arrived in SDK 0.7.0.

In [5]:
if not GUARDRAIL.exists():
    raise RuntimeError(
        "mode_b_shield_guardrail.py not found. Run this notebook from recipes/cost-controls/03-litellm/."
    )
WORK.mkdir(exist_ok=True)
shutil.copy(GUARDRAIL, WORK / "highflame_guardrail.py")
(WORK / "Dockerfile").write_text('''
FROM ghcr.io/berriai/litellm:main-stable

# The image has no pip. Bootstrap it into the image's venv, then add the Highflame SDK.
RUN python -m ensurepip --upgrade \\
 && python -m pip install --no-cache-dir 'highflame==0.10.0' 'python-dotenv>=1.0.0'

# The guardrail must be importable by the proxy process.
COPY highflame_guardrail.py /app/
ENV PYTHONPATH=/app
'''.lstrip())

rc, out = sh(f"docker build -t {IMAGE} {WORK}")
if rc != 0:
    print(out[-1500:])
    raise RuntimeError("docker build failed")
_, versions = sh(
    f"docker run --rm --entrypoint python {IMAGE} -c "
    "\"from importlib.metadata import version as v; print('litellm', v('litellm'), '| highflame', v('highflame'))\""
)
print("built", IMAGE, "->", versions)

built litellm-highflame-cost:demo -> litellm 1.101.0 | highflame 0.10.0


### The config: one added line

This is an ordinary Mode B config with one more line, `report_usage: true`. After each successful
call, LiteLLM hands the token counts to the guardrail, and the guardrail reports them to Highflame
with LiteLLM's call id, so a repeated report is never counted twice. The guardrail skips calls that
cost nothing at the provider (a LiteLLM cache hit, a `mock_response` route) and calls routed through
the Highflame AI Gateway, which reports them itself.

`mode: [pre_call]` is enough for a budget, because the budget is checked on the prompt. Keep your other
modes if you use the guardrail for more than cost.

In [6]:
(WORK / "config.yaml").write_text(f'''
model_list:
  - model_name: {MODEL}
    litellm_params:
      model: openai/{MODEL}
      api_key: os.environ/OPENAI_API_KEY

guardrails:
  - guardrail_name: highflame
    litellm_params:
      guardrail: highflame_guardrail.HighflameGuardrail
      mode: [pre_call]       # the budget is checked on the prompt, before the provider is called
      default_on: true       # REQUIRED. Without it, requests pass unguarded.
      report_usage: true     # report each call's token counts, so the budget counts them

general_settings:
  master_key: {MASTER_KEY}
'''.lstrip())
print((WORK / "config.yaml").read_text())

model_list:
  - model_name: gpt-4o
    litellm_params:
      model: openai/gpt-4o
      api_key: os.environ/OPENAI_API_KEY

guardrails:
  - guardrail_name: highflame
    litellm_params:
      guardrail: highflame_guardrail.HighflameGuardrail
      mode: [pre_call]       # the budget is checked on the prompt, before the provider is called
      default_on: true       # REQUIRED. Without it, requests pass unguarded.
      report_usage: true     # report each call's token counts, so the budget counts them

general_settings:
  master_key: sk-local-cost-demo



### Start the proxy

The proxy gets the Highflame key, the OpenAI key, and `HIGHFLAME_LOG_USAGE=1`, which logs every usage
report. The calls below read the price Highflame put on each call from that log.

In [7]:
def start_proxy() -> None:
    # Start the proxy and wait until it answers its liveness probe.
    sh(f"docker rm -f {CONTAINER}")
    env = {
        "HIGHFLAME_API_KEY": HIGHFLAME_API_KEY,
        "HIGHFLAME_BASE_URL": API_URL,
        "HIGHFLAME_TOKEN_URL": f"{AUTH_URL}/oauth2/token",
        "OPENAI_API_KEY": PROVIDER_API_KEY,
        "HIGHFLAME_LOG_USAGE": "1",  # log each usage report; the calls below read the price from it
    }
    envfile = WORK / ".runtime.env"
    envfile.touch(mode=0o600)
    envfile.chmod(0o600)
    envfile.write_text("".join(f"{name}={value}\n" for name, value in env.items()))
    rc, out = sh(
        f"docker run -d --name {CONTAINER} -p {PROXY_PORT}:4000 --env-file {envfile} "
        f"-v {WORK / 'config.yaml'}:/app/config.yaml:ro {IMAGE} --config /app/config.yaml --port 4000"
    )
    if rc != 0:
        raise RuntimeError(out)
    for second in range(1, 91):
        if sh(f"curl -fsS {PROXY_URL}/health/liveliness")[0] == 0:
            print(f"proxy up on {PROXY_URL} after {second}s")
            return
        time.sleep(1)
    print(sh(f"docker logs --tail 40 {CONTAINER}")[1])
    raise RuntimeError("the proxy did not become healthy")


start_proxy()

proxy up on http://localhost:4001 after 7s


### One call through the proxy

The client is a plain HTTP request to the proxy's OpenAI-compatible endpoint, as any app behind the
proxy would send. There is no Highflame code in it. The request carries a `litellm_session_id`, so
the guard checks and the usage reports all name the same session.

Every prompt below starts with a unique line. That keeps the provider's prompt cache out of the way,
so each call costs about the same.

In [8]:
# A support backlog to summarise: about 3,000 tokens of input, so each gpt-4o call costs about a cent.
ISSUES = ["login fails", "invoice is wrong", "export times out", "app crashes on start", "password reset email never arrives"]
PRODUCTS = ["the mobile app", "the web portal", "the billing page", "the admin console"]
BACKLOG = "\n".join(
    f"Ticket {n}: a customer reports that {ISSUES[n % 5]} in {PRODUCTS[n % 4]} since the last release."
    for n in range(1, 151)
)
TASK = "Write a short paragraph on each of the five most common problems in these tickets, with how many tickets mention it."


def work_prompt(step: int) -> str:
    return f"Request {step} of run {RUN_ID}.\n\n{TASK}\n\n{BACKLOG}"


SESSION = f"cost-demo-{RUN_ID}"  # one LiteLLM session for the run; the usage reports carry it too


def reported_cost(call_id: str) -> float | None:
    # The guardrail reports a call just after the proxy answers it, and logs the price Highflame
    # put on it. Wait for that line, for up to 10 seconds.
    for _ in range(20):
        _, line = sh(f"docker logs {CONTAINER} 2>&1 | grep 'usage reported:.*call_id={call_id} '")
        if "cost_usd=" in line:
            return float(line.rsplit("cost_usd=", 1)[1].split()[0])
        time.sleep(0.5)
    return None


def litellm_call(step: int) -> dict:
    global CALLS_MADE
    body = {
        "model": MODEL,
        "messages": [{"role": "user", "content": work_prompt(step)}],
        "max_tokens": 400,
        "litellm_session_id": SESSION,
    }
    request = urllib.request.Request(
        f"{PROXY_URL}/v1/chat/completions",
        data=json.dumps(body).encode(),
        headers={"Authorization": f"Bearer {MASTER_KEY}", "Content-Type": "application/json"},
    )
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            call_id = response.headers.get("x-litellm-call-id", "")
            reply = json.loads(response.read())
    except urllib.error.HTTPError as err:
        # The guardrail refuses with an HTTP 400 that carries Highflame's reason.
        raw = err.read().decode(errors="replace")
        try:
            detail = json.loads(raw).get("error", {}).get("message", raw)
        except ValueError:
            detail = raw
        return {"allowed": False, "tokens": "-", "cost": "-", "text": f"HTTP {err.code}: {detail}"}

    usage = reply.get("usage") or {}
    cost = reported_cost(call_id)
    CALLS_MADE += cost is not None
    return {
        "allowed": True,
        "tokens": f"{usage.get('prompt_tokens')}/{usage.get('completion_tokens')}",
        "cost": f"${cost:.4f}" if cost is not None else "not reported",
        "text": reply["choices"][0]["message"]["content"] or "",
    }


result = litellm_call(1)
print("allowed:", result["allowed"], " tokens in/out:", result["tokens"], " priced by Highflame:", result["cost"])
print("reply  :", result["text"][:200].replace("\n", " "), "...")
print("\nproject today:", meter(budget_state()))

allowed: True  tokens in/out: 3102/299  priced by Highflame: $0.0107
reply  : After analyzing the tickets, five recurring problems have emerged. The most common issue involves incorrect invoices, which is mentioned in 30 tickets. This problem spans multiple areas, including the ...



project today: [####................]  21% of $0.05 (~$0.011)


### What the guardrail reported

One line per report: the model, the token counts, LiteLLM's session and call ids, and the price
Highflame returned.

In [9]:
_, logs = sh(f"docker logs {CONTAINER} 2>&1 | grep '\\[highflame\\] usage'")
print(logs)

[highflame] usage reported: openai/gpt-4o in=3102 out=299 cached=0 session_id=cost-demo-3de8f7 call_id=b0232f72-1553-4788-bdce-f7a9c0936b7a priced=True cost_usd=0.010745


## 3. Keep going until the budget is used up

Each call adds to the project's spend. Once the bar reaches 100%, the next prompt is blocked.

In [10]:
refusal = None
previous = budget_state()
for step in range(2, 31):  # stops at the first refusal; the cap only bounds a misconfigured run
    result = litellm_call(step)
    time.sleep(2)  # let the usage report land before the next read
    state = budget_state()
    print(
        f"{step:>2}  {'allowed' if result['allowed'] else 'BLOCKED':<8}"
        f"  tokens {result['tokens']:<9}  {result['cost']:<24}  {meter(state)}"
    )
    if not result["allowed"]:
        refusal = result["text"]
        break
    if previous.get("budget_exceeded_daily"):
        print(
            "\nThe budget was already used up, but this call was allowed. Check that the Daily Budget"
            " template is deployed in Enforce mode, not Monitor."
        )
        break
    previous = state

print("\nWhat the agent got back:\n ", (refusal or "(not blocked)")[:300])

 2  allowed   tokens 3102/400   $0.0118                   [#########...........]  45% of $0.05 (~$0.022)


 3  allowed   tokens 3102/400   $0.0118                   [#############.......]  68% of $0.05 (~$0.034)


 4  allowed   tokens 3102/400   $0.0118                   [##################..]  92% of $0.05 (~$0.046)


 5  allowed   tokens 3102/400   $0.0118                   [####################] 100% of $0.05 (~$0.050)  USED UP


 6  BLOCKED   tokens -          -                         [####################] 100% of $0.05 (~$0.050)  USED UP

What the agent got back:
  HTTP 400: Highflame policy denied this prompt: Enterprise Policies Triggered: Daily Budget


The guardrail's `pre_call` check refused the prompt **before** LiteLLM called OpenAI, so the refused
prompt cost nothing. The client got an HTTP 400 with Highflame's reason, the same as for any other
guardrail refusal.

## 4. Where the money went

Highflame keeps one record per priced call, and serves it as a chargeback report grouped by agent,
by user, or by model. Studio shows the same report under **Observatory → Chargeback**.

The report is built from a stream, so the newest calls can take a minute or two to appear. The cell
waits until every call this run made is in it. The report covers whole UTC days, so the rows below
are today's totals; the first line is this run alone.

In [11]:
for _ in range(24):  # up to about 4 minutes
    by_agent = chargeback("agent")
    if by_agent.get("total_calls", 0) >= BASELINE.get("total_calls", 0) + CALLS_MADE:
        break
    time.sleep(10)


def show(title: str, report: dict) -> None:
    print(title)
    for row in report.get("rows", []):
        if row["cost_usd"] > 0:  # calls at $0 (an unpriced model, for one) carry no cost to explain
            label = row["key"] or "(not a registered agent)"
            print(f"  {label:<28} ${row['cost_usd']:.4f}  {row['calls']:>3} calls  {row['share']:.0%}")


run_cost = by_agent.get("total_cost_usd", 0) - BASELINE.get("total_cost_usd", 0)
print(f"This run: ${run_cost:.4f} over {CALLS_MADE} calls")
print(f"Today, this project: ${by_agent.get('total_cost_usd', 0):.4f} over {by_agent.get('total_calls', 0)} calls\n")
show("By agent (today)", by_agent)
show("\nBy model (today)", chargeback("model"))

This run: $0.0578 over 5 calls
Today, this project: $0.0578 over 5 calls

By agent (today)
  litellm-proxy                $0.0578    5 calls  100%



By model (today)
  openai/gpt-4o                $0.0578    5 calls  100%


## Clean up, and run it again

The cell below removes the proxy container and the build directory, which holds the proxy's env file
with your keys.

The project stays blocked for the rest of the UTC day. To run the notebook again today, raise the
budget in Studio (**Cost Controls → Edit budget**). To turn budgets off, remove the budget: with no
budget, the Cost Controls templates never fire.

In [12]:
sh(f"docker rm -f {CONTAINER}")
shutil.rmtree(WORK, ignore_errors=True)  # it held .runtime.env, which has your keys
print("removed the proxy container and", WORK.name)
print("the image stays; remove it with: docker rmi", IMAGE)

removed the proxy container and _highflame_litellm_cost
the image stays; remove it with: docker rmi litellm-highflame-cost:demo


## What this showed

- A daily budget in US dollars stopped the model calls of a LiteLLM proxy that calls OpenAI directly,
  once the budget was used up. The apps behind the proxy did not change.
- The guardrail needed one line of config, `report_usage: true`. Highflame priced every call from one
  price table and returned the price.
- The guardrail's `pre_call` check refused the next prompt before it reached OpenAI.
- The chargeback report showed the spend by agent and by model.

**Good to know**

- Spend is known only after a model answers, so the call that crosses the budget completes. The
  Monthly Budget Headroom template blocks at 95% to keep the overshoot small.
- A model that is not in Highflame's price table is counted at $0.
- Days and months are UTC.
- A project has one budget. Calls through the AI Gateway in the same project count against it too.
- The spend belongs to the Highflame key the proxy signs with, not to the callers behind the proxy.
  For limits per team or per developer, keep using LiteLLM's own per-key budgets beside the project
  budget.